In [24]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from mlxtend.frequent_patterns import apriori, association_rules

# 1. LOAD THE DATA 

In [25]:
print("Loading data and matrices")
df = pd.read_csv('data/preprocessed/cleaned_retail_data_2022_2025.csv')
user_item_matrix = pd.read_pickle('data/preprocessed/user_item_matrix.pkl')
b2c_basket = pd.read_pickle('data/preprocessed/basket_matrix_b2c.pkl')
b2b_basket= pd.read_pickle('data/preprocessed/basket_matrix_b2b.pkl')

Loading data and matrices


# 2. POPULARITY BASELINE

In [26]:
# In retail, "popularity" is usually better measured by how MANY times an item 
# was bought (Frequency) rather than total quantity, to avoid bulk-buy skew.
def train_popularity_model(data, top_n=10):
    """
    Calculates the most popular products based on the number of unique invoices.
    """
    popularity = data.groupby('ProductName')['InvoiceID'].nunique().reset_index()
    popularity.columns = ['ProductName', 'PurchaseFrequency']
    popularity = popularity.sort_values(by='PurchaseFrequency', ascending=False)
    
    return popularity.head(top_n)

top_popular_items = train_popularity_model(df, top_n=10)
print("\n--- POPULARITY BASELINE ---")
print("Top 5 Most Popular Products:")
print(top_popular_items.head(5).to_string(index=False))



--- POPULARITY BASELINE ---
Top 5 Most Popular Products:
                                                                   ProductName  PurchaseFrequency
                                 CEZAR KRZYZYKI ANEKS DO GLAZURY 1.5-6MM PROF.               1003
                        HERMES PAPIER SCIERNY NA PLOTNIE 23X28 /SIATKA SCIERNA                985
MAPEI ZAPRAWA KLEJOWA KERAFLEX. **EXTRA** S1 SZARY / ZERO 25KG ***PROMOCJA***-                929
                     MAPEI ZAPRAWA KLEJOWA ADESILEX P9 ...25KG (paleta 50 szt)                868
                             HERMES PEDZEL PIERSCIENIOWY METAL.25 LUX (03 240)                708


# 3. ITEM-BASED COLLABORATIVE FILTERING

In [27]:
# We calculate the Cosine Similarity between items. 
# If Item A and Item B are frequently bought by the same contractors, 
# their cosine similarity score will be close to 1.

print("\nCalculating Item Similarity Matrix (this might take a moment)...")
# Transpose the matrix so products are rows and customers are columns
item_user_matrix = user_item_matrix.T 

# Calculate cosine similarity between items
item_sim_array = cosine_similarity(item_user_matrix)

# Convert the resulting array back into a readable DataFrame
item_similarity_df = pd.DataFrame(
    item_sim_array, 
    index=user_item_matrix.columns, 
    columns=user_item_matrix.columns
)

def get_item_similarity_recommendations(product_name, similarity_matrix, top_n=5):
    """
    Given a product name, returns the top N most similar products.
    """
    if product_name not in similarity_matrix.columns:
        return f"Product '{product_name}' not found in the matrix."
    
    # Get similarity scores for the target product
    similar_scores = similarity_matrix[product_name]
    
    # Sort scores in descending order, drop the product itself (similarity = 1.0)
    similar_items = similar_scores.sort_values(ascending=False).drop(product_name)
    
    return similar_items.head(top_n)

# Example Usage: Let's pick a specific product from your data
# (Replace with an actual product name from your dataset to test)
sample_product = user_item_matrix.columns[0] # Just picking the first product alphabetically
print(f"\n--- ITEM SIMILARITY MODEL ---")
print(f"Because customers bought: {sample_product}")
print("They also bought:")
print(get_item_similarity_recommendations(sample_product, item_similarity_df, top_n=5))




Calculating Item Similarity Matrix (this might take a moment)...

--- ITEM SIMILARITY MODEL ---
Because customers bought: .DURO SYSTEM KSZTAŁ. ZAPR. LIST. BAT. 150MM Z KOL. PRZYŚĆ. 16X1/2 D/S-412-LB1501612
They also bought:
ProductName
LIPTON TEKTURA BUD 15M2                                           1.0
ROCA ONA ZESTAW WC MISKA WC PODW. RIM+DESKA W/OPAD  A34H689000    1.0
FASTER TOOLS PRZYSTAWKA KATOWA Z BITAMI 6975                      1.0
.DURO-401 ZŁĄCZKA NAKRĘTNA 16X1/2" GW D/S-401-ZGW1612             1.0
FRANSPOL GOTOWA MASA SZPACHLOWA SPECJAL 5KG (BEZ TASMY)           1.0
Name: .DURO SYSTEM KSZTAŁ. ZAPR. LIST. BAT. 150MM Z KOL. PRZYŚĆ. 16X1/2 D/S-412-LB1501612, dtype: float64


# 4. ASSOCIATION RULES (Apriori)

In [28]:
#  TRAIN THE APRIORI MODELS
def generate_association_rules(basket_matrix, min_support=0.005, min_lift=1.2):
    """
    Generates association rules from a boolean basket matrix.
    Note: Offline retail data is very sparse, so min_support must be low (e.g., 0.5%).
    """
    # Find frequent itemsets
    frequent_itemsets = apriori(basket_matrix, min_support=min_support, use_colnames=True)
    
    # If no frequent itemsets are found, return empty dataframe
    if frequent_itemsets.empty:
        print("Warning: No frequent itemsets found. Try lowering min_support.")
        return pd.DataFrame()
        
    # Generate rules
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=min_lift)
    
    # Sort by confidence and lift (the most actionable metrics)
    rules = rules.sort_values(['lift', 'confidence'], ascending=[False, False])
    
    # Clean up the format for readability
    rules['antecedents'] = rules['antecedents'].apply(lambda x: list(x)[0])
    rules['consequents'] = rules['consequents'].apply(lambda x: list(x)[0])
    
    return rules


* B2C Model

In [29]:
print("\nGenerating Rules for B2C Customers")
b2c_rules = generate_association_rules(b2c_basket, min_support=0.005) # 0.5% of all B2C invoices
print(f"Found {len(b2c_rules)} B2C rules.")
if not b2c_rules.empty:
    print(b2c_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5).to_string(index=False))


Generating Rules for B2C Customers
Found 0 B2C rules.


* B2B Model

In [30]:
# --- Run B2B Model ---
print("\nGenerating Rules for B2B Customers")
b2b_rules = generate_association_rules(b2b_basket, min_support=0.005) # 0.5% of all B2B invoices
print(f"Found {len(b2b_rules)} B2B rules.")
if not b2b_rules.empty:
    print(b2b_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5).to_string(index=False))


Generating Rules for B2B Customers
Found 94 B2B rules.
                     antecedents                      consequents  support  confidence      lift
      AKCESS KOLANKO 50/15 11011 AKCESS KOLANKO 50/30 11146/11061 0.005901    0.625000 61.643519
AKCESS KOLANKO 50/30 11146/11061       AKCESS KOLANKO 50/15 11011 0.005901    0.582011 61.643519
AKCESS KOLANKO 50/30 11146/11061 AKCESS KOLANKO 50/45 11145/11013 0.006384    0.629630 58.392666
AKCESS KOLANKO 50/45 11145/11013 AKCESS KOLANKO 50/30 11146/11061 0.006384    0.592040 58.392666
      AKCESS KOLANKO 50/15 11011 AKCESS KOLANKO 50/45 11145/11013 0.005740    0.607955 56.382491


# 5. EXPORT

In [ ]:
# Save the similarity matrix for the Hybrid Model later
item_similarity_df.to_pickle('data/preprocessed/item_similarity_matrix.pkl')
b2c_rules.to_csv('data/preprocessed/b2c_association_rules.csv', index=False)
b2b_rules.to_csv('data/preprocessed/b2b_association_rules.csv', index=False)
print("Files saved successfully.")


<>:5: SyntaxWarning: invalid escape sequence '\F'
<>:5: SyntaxWarning: invalid escape sequence '\F'
/var/folders/gm/vll5hdf55pq69ynclc1r1f0m0000gn/T/ipykernel_87974/1795197568.py:5: SyntaxWarning: invalid escape sequence '\F'
  print("\Files saved successfully.")


\Files saved successfully.


# 6. MODEL EVALUATION (Leave-One-Out)
Precision@K, Recall@K and NDCG@K are computed for each model using a leave-one-out strategy on B2B customers: the most recent invoice is held out as the test set, and the remaining purchase history is used for training.


In [ ]:
import numpy as np

def precision_at_k(recommended, relevant, k):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / len(relevant)

def ndcg_at_k(recommended, relevant, k):
    gains = [1 if item in relevant else 0 for item in recommended[:k]]
    dcg   = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    idcg  = sum(1 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_models(data, item_sim_matrix, b2b_rules_df, top_k=5, sample_n=100):
    """
    Leave-one-out evaluation on Known B2B customers.
    Holds out each customer's most recent invoice and checks whether
    Popularity, Item-CF, Association Rules, and the Hybrid model
    would have retrieved those items in their top-K recommendations.
    """
    known = data[data['CustomerName'] != 'SPRZEDAZ FISKALNA'].copy()

    # Customers with at least 2 invoices (1 for training, 1 for testing)
    inv_counts = known.groupby('CustomerName')['InvoiceID'].nunique()
    valid_customers = inv_counts[inv_counts >= 2].index.tolist()

    if len(valid_customers) > sample_n:
        np.random.seed(42)
        valid_customers = list(np.random.choice(valid_customers, sample_n, replace=False))

    results = {'Popularity': [], 'Item-CF': [], 'Assoc. Rules': [], 'Hybrid': []}
    popularity_list = (
        data.groupby('ProductName')['InvoiceID'].nunique()
        .sort_values(ascending=False).index.tolist()
    )

    for customer in valid_customers:
        cust_data = known[known['CustomerName'] == customer].sort_values('SaleDate')
        invoices  = cust_data['InvoiceID'].unique()

        last_invoice = invoices[-1]
        test_items  = cust_data[cust_data['InvoiceID'] == last_invoice]['ProductName'].tolist()
        train_items = cust_data[cust_data['InvoiceID'] != last_invoice]['ProductName'].unique().tolist()

        if not test_items or not train_items:
            continue

        # 1. Popularity baseline
        pop_recs = [p for p in popularity_list if p not in train_items][:top_k]

        # 2. Item-Based CF
        cf_scores = {}
        for item in train_items:
            if item in item_sim_matrix.columns:
                for rec_item, score in item_sim_matrix[item].items():
                    if score > 0 and rec_item not in train_items:
                        cf_scores[rec_item] = max(cf_scores.get(rec_item, 0), score)
        cf_recs = sorted(cf_scores, key=cf_scores.get, reverse=True)[:top_k]

        # 3. Association Rules
        ar_seen = {}
        if not b2b_rules_df.empty:
            for item in train_items:
                for _, row in b2b_rules_df[b2b_rules_df['antecedents'] == item].iterrows():
                    rec_item = row['consequents']
                    if rec_item not in train_items and rec_item not in ar_seen:
                        ar_seen[rec_item] = row['lift']
        ar_recs = sorted(ar_seen, key=ar_seen.get, reverse=True)[:top_k]

        # 4. Hybrid (CF + AR with popularity fallback)
        hybrid_recs = list(dict.fromkeys(cf_recs + ar_recs)) or pop_recs
        hybrid_recs = hybrid_recs[:top_k]

        for model_name, recs in [('Popularity', pop_recs), ('Item-CF', cf_recs),
                                  ('Assoc. Rules', ar_recs), ('Hybrid', hybrid_recs)]:
            if recs:
                results[model_name].append({
                    'P@K':    precision_at_k(recs, test_items, top_k),
                    'R@K':    recall_at_k(recs, test_items, top_k),
                    'NDCG@K': ndcg_at_k(recs, test_items, top_k),
                })

    summary = {}
    for model_name, scores in results.items():
        if scores:
            summary[model_name] = {
                f'Precision@{top_k}': round(np.mean([s['P@K']    for s in scores]), 4),
                f'Recall@{top_k}':    round(np.mean([s['R@K']    for s in scores]), 4),
                f'NDCG@{top_k}':      round(np.mean([s['NDCG@K'] for s in scores]), 4),
            }
    return pd.DataFrame(summary).T

K = 5
print(f"Running leave-one-out evaluation at K={K} (sampling up to 100 B2B customers)...")
eval_results = evaluate_models(df, item_similarity_df, b2b_rules, top_k=K, sample_n=100)
print("\nModel Comparison:")
print(eval_results.to_string())

eval_results.to_csv('data/preprocessed/model_evaluation_results.csv')
print("\nEvaluation results saved to data/preprocessed/model_evaluation_results.csv")
